# RQ1 Box Plots — Sentiment Distribution by Profile & Dimension

**Project:** MLLM Persona Simulation — UrbCom 2026  
**Input:** `outputs/annotations_baseline.jsonl`  
**Purpose:** Produce Figure 6 and Figure 7 of the paper — per-profile and
per-dimension sentiment box plots — and the parse/retry failure analysis.

---


---
## 0  Environment

In [ ]:
import sys
from pathlib import Path

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd().parent  # notebook lives in notebooks/

sys.path.insert(0, str(ROOT / "src"))

import json
import warnings
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", font_scale=1.1)

FIGURES_DIR = ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# Paths
ANNOTATIONS_BASELINE = ROOT / "outputs" / "annotations_baseline.jsonl"
FAILURES = ROOT / "outputs" / "annotation_failures.jsonl"
PERSONAS_BASELINE = ROOT / "outputs" / "personas_baseline.jsonl"
DATASET_JSON = ROOT / "data" / "perceptsent-raw" / "dataset.json"

print(f"ROOT: {ROOT}")

---
## 1  Data Loading

In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    """Load a JSONL file into a DataFrame."""
    records = []
    if not path.exists():
        print(f"WARNING: {path} not found")
        return pd.DataFrame()
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


# --- Baseline annotations ---
df_ann = load_jsonl(ANNOTATIONS_BASELINE)
print(f"Baseline annotations: {len(df_ann):,}")

# --- Failures ---
df_fail = load_jsonl(FAILURES)
print(f"Annotation failures: {len(df_fail):,}")

# --- Full persona annotations (for cross-condition) ---
print(f"Full persona annotations: {len(df_full):,}")

# --- Persona metadata (persona_id → profile_id) ---
df_personas = load_jsonl(PERSONAS_BASELINE)
print(f"Personas: {len(df_personas):,}")

---
## 2  Sentiment Mapping & Profile Assignment

In [ ]:
# Ordinal sentiment mapping
SENT_MAP = {
    "Negative": -2,
    "SlightlyNegative": -1,
    "Neutral": 0,
    "SlightlyPositive": 1,
    "Positive": 2,
}
SENT_LABELS = list(SENT_MAP.keys())  # ordered
SENT_COLORS = {
    -2: "#d62728",   # red
    -1: "#ff7f0e",   # orange
     0: "#7f7f7f",   # grey
     1: "#2ca02c",   # green
     2: "#1f77b4",   # blue
}

# Map sentiment to numeric
df_ann["sentiment_num"] = df_ann["predicted_sentiment"].map(SENT_MAP)
df_ann = df_ann.dropna(subset=["sentiment_num"])  # drop any empty/invalid
df_ann["sentiment_num"] = df_ann["sentiment_num"].astype(int)

# Extract demographics from raw_demographics dict column
if isinstance(df_ann["raw_demographics"].iloc[0], str):
    df_ann["raw_demographics"] = df_ann["raw_demographics"].apply(json.loads)

demo_df = pd.json_normalize(df_ann["raw_demographics"])
for col in ["gender", "economic_status", "political_spectrum", "personality"]:
    df_ann[col] = demo_df[col].values

# Assign profile_id from personas file
pid_to_profile = dict(zip(df_personas["persona_id"], df_personas["profile_id"]))
df_ann["profile_id"] = df_ann["persona_id"].map(pid_to_profile)

# Create short profile label
GENDER_SHORT = {"Male": "M", "Female": "F"}
ECON_SHORT = {"Low income": "Low", "High income": "High"}
POL_SHORT = {"Progressive": "Prog", "Conservative": "Cons"}
PERS_SHORT = {"Analytical": "Ana", "Empathetic": "Emp", "Pragmatic": "Prag"}

df_ann["profile_label"] = (
    df_ann["gender"].map(GENDER_SHORT) + "·"
    + df_ann["economic_status"].map(ECON_SHORT) + "·"
    + df_ann["political_spectrum"].map(POL_SHORT) + "·"
    + df_ann["personality"].map(PERS_SHORT)
)

print(f"Unique profiles in annotations: {df_ann['profile_id'].nunique()}")
print(f"Unique personas: {df_ann['persona_id'].nunique()}")
print(f"Unique images: {df_ann['image_id'].nunique()}")
print(f"\nSentiment distribution (numeric):")
print(df_ann["sentiment_num"].value_counts().sort_index())

---
## 3  RQ1 Box Plots — Per Profile

One box per demographic profile showing the distribution of numerical sentiment scores.  
Colour = modal sentiment of that profile.

In [ ]:
# Compute modal sentiment per profile for colouring
profile_modal = (
    df_ann.groupby("profile_label")["sentiment_num"]
    .agg(lambda x: x.mode().iloc[0])
    .sort_values()
)

# Order profiles by modal sentiment, then alphabetically within
profile_order = profile_modal.index.tolist()
modal_colors = [SENT_COLORS[profile_modal[p]] for p in profile_order]

fig, ax = plt.subplots(figsize=(18, 7))
bp = sns.boxplot(
    data=df_ann,
    x="profile_label",
    y="sentiment_num",
    order=profile_order,
    palette=modal_colors,
    showfliers=False,
    width=0.6,
    ax=ax,
)

# ── Annotate each box with statistics ─────────────────────────────────────────
for i, profile in enumerate(profile_order):
    vals = df_ann.loc[df_ann["profile_label"] == profile, "sentiment_num"]
    q1 = vals.quantile(0.25)
    med = vals.median()
    q3 = vals.quantile(0.75)
    mean = vals.mean()
    n = len(vals)
    modal_ratio = vals.value_counts().max() / n

    # Median label (inside the box, centered)
    ax.text(i, med + 0.08, f"med={med:.0f}", ha="center", va="bottom",
            fontsize=7, fontweight="bold", color="black",
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1))
    # Q1 and Q3 labels (at the box edges)
    ax.text(i + 0.32, q1, f"Q1={q1:.1f}", ha="left", va="center",
            fontsize=6, color="#444444")
    ax.text(i + 0.32, q3, f"Q3={q3:.1f}", ha="left", va="center",
            fontsize=6, color="#444444")
    # Mean marker + label
    ax.plot(i, mean, marker="D", color="black", markersize=4, zorder=5)
    ax.text(i - 0.32, mean, f"μ={mean:.2f}", ha="right", va="center",
            fontsize=6, color="#222222", fontstyle="italic")
    # N and modal ratio below the box
    ax.text(i, -2.45, f"n={n:,}\nMR={modal_ratio:.2f}", ha="center", va="top",
            fontsize=6, color="#555555")

ax.set_xlabel("Demographic Profile", fontsize=12)
ax.set_ylabel("Sentiment Score", fontsize=12)
ax.set_yticks([-2, -1, 0, 1, 2])
ax.set_yticklabels(["Negative\n(−2)", "Sl. Negative\n(−1)", "Neutral\n(0)",
                     "Sl. Positive\n(+1)", "Positive\n(+2)"])
ax.set_ylim(-2.7, 2.5)
plt.xticks(rotation=45, ha="right", fontsize=9)

# Legend for modal colours
legend_patches = [
    mpatches.Patch(color=SENT_COLORS[v], label=k)
    for k, v in SENT_MAP.items()
]
ax.legend(handles=legend_patches, title="Modal Sentiment", loc="upper left",
          fontsize=8, title_fontsize=9)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "rq1_boxplot_per_profile.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "rq1_boxplot_per_profile.png", bbox_inches="tight", dpi=300)
print("Saved: rq1_boxplot_per_profile.pdf/.png")
plt.show()

---
## 4  RQ1 Box Plots — Per Demographic Dimension

Four facets: gender, economic_status, political_spectrum, personality.  
Each subplot shows the sentiment distribution for each category value.

In [ ]:
DIMENSIONS = [
    ("gender", ["Male", "Female"]),
    ("economic_status", ["Low income", "High income"]),
    ("political_spectrum", ["Progressive", "Conservative"]),
    ("personality", ["Analytical", "Empathetic", "Pragmatic"]),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 6), sharey=True)

for ax, (dim, cats) in zip(axes, DIMENSIONS):
    sns.boxplot(
        data=df_ann,
        x=dim,
        y="sentiment_num",
        order=cats,
        palette="Set2",
        showfliers=False,
        width=0.5,
        ax=ax,
    )

    # ── Annotate each box with statistics ─────────────────────────────────
    for j, cat in enumerate(cats):
        vals = df_ann.loc[df_ann[dim] == cat, "sentiment_num"]
        q1 = vals.quantile(0.25)
        med = vals.median()
        q3 = vals.quantile(0.75)
        mean = vals.mean()
        n = len(vals)
        modal_ratio = vals.value_counts().max() / n

        # Median label
        ax.text(j, med + 0.10, f"med={med:.0f}", ha="center", va="bottom",
                fontsize=7, fontweight="bold",
                bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1))
        # Q1 / Q3
        ax.text(j + 0.28, q1, f"Q1={q1:.1f}", ha="left", va="center",
                fontsize=6, color="#444444")
        ax.text(j + 0.28, q3, f"Q3={q3:.1f}", ha="left", va="center",
                fontsize=6, color="#444444")
        # Mean diamond
        ax.plot(j, mean, marker="D", color="black", markersize=4, zorder=5)
        ax.text(j - 0.28, mean, f"μ={mean:.2f}", ha="right", va="center",
                fontsize=6, color="#222222", fontstyle="italic")
        # N and modal ratio
        ax.text(j, -2.55, f"n={n:,}\nMR={modal_ratio:.2f}", ha="center", va="top",
                fontsize=6, color="#555555")

    ax.set_xlabel(dim.replace("_", " ").title(), fontsize=11)
    ax.set_ylabel("" if ax != axes[0] else "Sentiment Score", fontsize=11)
    ax.set_yticks([-2, -1, 0, 1, 2])
    ax.set_ylim(-2.8, 2.5)
    if ax == axes[0]:
        ax.set_yticklabels(["−2", "−1", "0", "+1", "+2"])

plt.tight_layout()
fig.savefig(FIGURES_DIR / "rq1_boxplot_per_dimension.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "rq1_boxplot_per_dimension.png", bbox_inches="tight", dpi=300)
print("Saved: rq1_boxplot_per_dimension.pdf/.png")
plt.show()

---
## 5  Parse Retry & Failure Analysis

In [ ]:
# Combine successful + failed annotations for retry analysis
retry_success = df_ann["parse_retries"].value_counts().sort_index()
retry_fail = df_fail["parse_retries"].value_counts().sort_index() if len(df_fail) > 0 else pd.Series(dtype=int)

# Merge into a single DataFrame for plotting
retry_df = pd.DataFrame({
    "Successful": retry_success,
    "Failed (exhausted)": retry_fail,
}).fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Retry distribution bar chart ---
retry_df.plot(kind="bar", stacked=True, ax=axes[0],
              color=["#2ca02c", "#d62728"], edgecolor="white")
axes[0].set_xlabel("Parse Retries", fontsize=11)
axes[0].set_ylabel("Count", fontsize=11)
axes[0].tick_params(axis="x", rotation=0)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%d", fontsize=8)

# --- Right: Failure breakdown by image_id ---
if len(df_fail) > 0:
    fail_by_img = df_fail["image_id"].value_counts().head(10)
    fail_by_img.plot(kind="barh", ax=axes[1], color="#d62728", edgecolor="white")
    axes[1].set_xlabel("Failure Count", fontsize=11)
    axes[1].set_ylabel("Image ID (truncated)", fontsize=11)
    # Truncate long image IDs for readability
    axes[1].set_yticklabels([t[:20] + "..." if len(t) > 20 else t
                             for t in fail_by_img.index], fontsize=8)
    axes[1].bar_label(axes[1].containers[0], fmt="%d", fontsize=9)
else:
    axes[1].text(0.5, 0.5, "No failures recorded", ha="center", va="center",
                 fontsize=12, transform=axes[1].transAxes)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "parse_retry_distribution.pdf", bbox_inches="tight", dpi=300)
fig.savefig(FIGURES_DIR / "parse_retry_distribution.png", bbox_inches="tight", dpi=300)
print("Saved: parse_retry_distribution.pdf/.png")

# Summary stats
total_attempts = len(df_ann) + len(df_fail)
print(f"\nTotal annotation attempts: {total_attempts:,}")
print(f"Successful: {len(df_ann):,} ({len(df_ann)/total_attempts*100:.1f}%)")
print(f"Failed: {len(df_fail):,} ({len(df_fail)/total_attempts*100:.2f}%)")
print(f"Retries > 0: {(df_ann['parse_retries'] > 0).sum():,}")
plt.show()

---
## Outputs

Figures saved to `figures/`:
- `rq1_boxplot_per_profile.pdf` — per-profile sentiment distribution (Fig. 7)
- `rq1_boxplot_per_dimension.pdf` — per-dimension sentiment distribution (Fig. 6)
- `parse_retry_distribution.pdf` — annotation failure analysis (Fig. 3)
